In [99]:
import pandas as pd

In [100]:
data = pd.read_csv('../Dataset/data-en-hi-de-fr.csv')
data

,labels,text,text_hi,text_de,text_fr
0,ham,"Go until jurong point, crazy.. Available only ...","Dakag बिंदु तक जाओ, पागल. केवल Bag Non महान वि...","Gehen Sie bis jurong Punkt, verrückt.. Verfügb...","Allez jusqu'à Jurong point, fou.. Disponible s..."
1,ham,Ok lar... Joking wif u oni...,ओके लामर.... if if uue पर.,Ok Lar... joking wif u oni...,J'ai fait une blague sur le wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,Fktatatat 21 मई को प्राप्त करने के लिए मुफ्त प...,Freier Eintritt in 2 a wkly comp zum Gewinn FA...,Entrée libre dans 2 a wkly comp pour gagner FA...
3,ham,U dun say so early hor... U c already then say...,Uden इतना जल्दी कहते हैं... तो पहले से ही यूसी...,U dun sagen so früh... U c schon dann sagen...,U dun dit si tôt hor... U c déjà dire alors...
4,ham,"Nah I don't think he goes to usf, he lives aro...","नहीं, मुझे नहीं लगता कि वह हमारे लिए चला जाता ...","Nein, ich glaube nicht, dass er zu unsf geht, ...","Non, je ne pense pas qu'il va à usf, il vit da..."
...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,यह 2 सेकंड है जब हमने 2 संपर्क की कोशिश की है....,"Dies ist das zweite Mal, dass wir versucht hab...",C'est la 2ème fois que nous avons essayé 2 con...
5568,ham,Will ü b going to esplanade fr home?,क्या कलाई घर का पता लगाने के लिए जा रही होगी?,"Wird u b gehen, um esplanade fr home?",Est-ce que ü b ira à l'esplanade en maison?
5569,ham,"Pity, * was in mood for that. So...any other s...","तो फिर, दूसरे सुझाव क्या हैं?","Schade, * war in Stimmung dafür. Also... irgen...","Dommage, * était d'humeur pour ça. Donc... d'a..."
5570,ham,The guy did some bitching but I acted like i'd...,आदमी कुछ कुतियािंग किया लेकिन मैं मैं कुछ और ख...,"Der Typ hat ein bisschen rumgeschnüffelt, aber...",Le type a fait une saloperie mais j'ai agi com...


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
data_combine = (data['text']+" "+data['text_hi']+" "+data['text_de']+" "+data['text_fr'])
vectorizer = CountVectorizer()
x = (data_combine)
y = data['labels']
x.toarray()

In [102]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(x,y,random_state=42,test_size=0.2)

In [103]:
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
model = MultinomialNB()
pipeline = Pipeline([
    ('tfidf',CountVectorizer(
        lowercase=True
    )),
    ('model',MultinomialNB())
])
pipeline.fit(x_train,y_train)

,steps,"[('tfidf', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [104]:
from sklearn.metrics import classification_report,confusion_matrix
y_pred = pipeline.predict(x_test)
print("========CounteVectorizer==============")
print(classification_report(y_test,y_pred))

========CounteVectorizer==============
              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       966
        spam       0.95      0.94      0.95       149

    accuracy                           0.99      1115
   macro avg       0.97      0.97      0.97      1115
weighted avg       0.99      0.99      0.99      1115



In [105]:
print(confusion_matrix(y_test,y_pred))
print(y_pred)

[[959   7]
 [  9 140]]
['ham' 'ham' 'ham' ... 'ham' 'ham' 'ham']


In [106]:
from sklearn.model_selection import cross_val_score,StratifiedKFold

skf = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scores = cross_val_score(pipeline,x,y,cv=skf,scoring='roc_auc')
print(scores.mean())
scores

0.9819422331953959


array([0.99383765, 0.98665976, 0.98422645, 0.96290295, 0.98208436])

In [107]:
message = [""]
y_pred_final = pipeline.predict_proba(message)[0][1]
if y_pred_final > 0.3:
    print("Spam")
else:
    print("No spam")

No spam
